# Байесовский классификатор спама

## Прикладной проект: Наивный байесовский фильтр электронной почты

В этом ноутбуке мы применим знания из **Модуля 1 (теорема Байеса)** и **Модуля 4 (MLE, байесовский вывод)** для построения классификатора спама с нуля.

### Идея наивного байесовского классификатора:

$$P(\text{spam} \mid \text{слова}) = \frac{P(\text{слова} \mid \text{spam})\,P(\text{spam})}{P(\text{слова})}$$

«Наивное» предположение — условная независимость слов при заданном классе.

### Содержание:
1. Генерация и загрузка данных
2. Exploratory Data Analysis (EDA)
3. Оценка вероятностей (MLE) со сглаживанием Лапласа
4. Реализация классификатора
5. Оценка качества: precision, recall, матрица ошибок
6. Апостериорные вероятности и интерпретация
7. Выводы

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Библиотеки успешно загружены!')

## 1. Генерация и загрузка данных

Сгенерируем синтетические письма. У спама и обычных писем (ham) разные вероятности появления слов.

### Структура данных:
- **text**: Текст письма (набор слов)
- **label**: Метка класса (`spam` / `ham`)

In [ ]:
# Генерация синтетических писем
np.random.seed(42)

# Словарь: вероятность слова в спаме и в ham
#                слово     P(word|spam)  P(word|ham)
vocab = {
    'выигрыш':      (0.30, 0.02),
    'бесплатно':    (0.35, 0.03),
    'кредит':       (0.25, 0.02),
    'скидка':       (0.28, 0.05),
    'срочно':       (0.22, 0.04),
    'деньги':       (0.30, 0.06),
    'приз':         (0.26, 0.02),
    'встреча':      (0.03, 0.25),
    'проект':       (0.04, 0.28),
    'отчёт':        (0.03, 0.24),
    'коллега':      (0.02, 0.20),
    'договор':      (0.05, 0.22),
    'привет':       (0.08, 0.30),
    'спасибо':      (0.06, 0.28),
    'вопрос':       (0.07, 0.26),
}
words = list(vocab.keys())

def generate_email(is_spam):
    idx = 0 if is_spam else 1
    tokens = [w for w in words if np.random.rand() < vocab[w][idx]]
    if not tokens:  # письмо не бывает пустым
        tokens = [np.random.choice(words)]
    return ' '.join(tokens)

n_spam, n_ham = 400, 600
data = []
for _ in range(n_spam):
    data.append({'text': generate_email(True), 'label': 'spam'})
for _ in range(n_ham):
    data.append({'text': generate_email(False), 'label': 'ham'})

df = pd.DataFrame(data).sample(frac=1, random_state=42).reset_index(drop=True)

print('Данные электронной почты')
print('=' * 60)
print(f'Всего писем: {len(df)}')
print(f'Спам: {(df["label"] == "spam").sum()}')
print(f'Ham:  {(df["label"] == "ham").sum()}')
print(f'\nПримеры писем:')
print(df.head(8).to_string(index=False))

## 2. Exploratory Data Analysis (EDA)

Посмотрим, какие слова наиболее характерны для спама и ham.

In [ ]:
# Подсчёт частот слов по классам
spam_words = Counter()
ham_words = Counter()

for _, row in df.iterrows():
    tokens = row['text'].split()
    if row['label'] == 'spam':
        spam_words.update(tokens)
    else:
        ham_words.update(tokens)

freq_df = pd.DataFrame({
    'spam': pd.Series(spam_words),
    'ham': pd.Series(ham_words)
}).fillna(0).astype(int)
freq_df = freq_df.loc[words]

print('Частота слов по классам:')
print(freq_df)

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

freq_df.sort_values('spam', ascending=True).plot.barh(
    y='spam', ax=axes[0], color='crimson', alpha=0.7, legend=False)
axes[0].set_xlabel('Частота в спаме')
axes[0].set_title('Частота слов в СПАМЕ')

freq_df.sort_values('ham', ascending=True).plot.barh(
    y='ham', ax=axes[1], color='seagreen', alpha=0.7, legend=False)
axes[1].set_xlabel('Частота в ham')
axes[1].set_title('Частота слов в HAM')

plt.tight_layout()
plt.show()

## 3. Оценка вероятностей (MLE) со сглаживанием Лапласа

Оценим параметры методом максимального правдоподобия. Чтобы избежать нулевых вероятностей для слов, не встретившихся в классе, применим **сглаживание Лапласа**:

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{N_c + \alpha \cdot |V|}$$

где $\alpha=1$ — параметр сглаживания, $|V|$ — размер словаря, $N_c$ — суммарное число слов класса $c$.

In [ ]:
# Разделение на обучающую и тестовую выборки
train = df.sample(frac=0.7, random_state=1)
test = df.drop(train.index).reset_index(drop=True)
train = train.reset_index(drop=True)
print(f'Обучающая выборка: {len(train)}, тестовая: {len(test)}')

# Априорные вероятности классов P(spam), P(ham)
p_spam = (train['label'] == 'spam').mean()
p_ham = (train['label'] == 'ham').mean()
print(f'\nАприорные вероятности:')
print(f'P(spam) = {p_spam:.3f}')
print(f'P(ham)  = {p_ham:.3f}')

# Подсчёт слов на обучающей выборке
sw = Counter()
hw = Counter()
for _, row in train.iterrows():
    tok = row['text'].split()
    (sw if row['label'] == 'spam' else hw).update(tok)

alpha = 1.0
V = len(words)
N_spam = sum(sw.values())
N_ham = sum(hw.values())

def word_prob(word, cls):
    if cls == 'spam':
        return (sw[word] + alpha) / (N_spam + alpha * V)
    return (hw[word] + alpha) / (N_ham + alpha * V)

prob_table = pd.DataFrame({
    'P(w|spam)': [word_prob(w, 'spam') for w in words],
    'P(w|ham)': [word_prob(w, 'ham') for w in words],
}, index=words)
prob_table['log_ratio'] = np.log(prob_table['P(w|spam)'] / prob_table['P(w|ham)'])
print('\nОценённые вероятности слов (MLE + сглаживание):')
print(prob_table.round(4))

## 4. Реализация классификатора

Чтобы избежать переполнения при перемножении малых вероятностей, работаем в **логарифмах**:

$$\log P(c \mid \text{слова}) \propto \log P(c) + \sum_i \log P(w_i \mid c)$$

In [ ]:
# Наивный байесовский классификатор в лог-пространстве
def classify(text, return_proba=False):
    tokens = [w for w in text.split() if w in vocab]
    log_spam = np.log(p_spam)
    log_ham = np.log(p_ham)
    for w in tokens:
        log_spam += np.log(word_prob(w, 'spam'))
        log_ham += np.log(word_prob(w, 'ham'))
    # Нормализация в вероятность через log-sum-exp
    m = max(log_spam, log_ham)
    p_s = np.exp(log_spam - m)
    p_h = np.exp(log_ham - m)
    proba_spam = p_s / (p_s + p_h)
    label = 'spam' if proba_spam >= 0.5 else 'ham'
    if return_proba:
        return label, proba_spam
    return label

# Примеры классификации
examples = [
    'выигрыш бесплатно приз деньги',
    'привет коллега отчёт проект',
    'скидка срочно кредит',
    'спасибо вопрос встреча',
]
print('Примеры классификации')
print('=' * 60)
for text in examples:
    label, proba = classify(text, return_proba=True)
    print(f'[{label.upper():4}] P(spam)={proba:.3f}  <- "{text}"')

## 5. Оценка качества

Оценим классификатор на тестовой выборке. Основные метрики:
- **Accuracy** — доля правильных ответов
- **Precision** — $\frac{TP}{TP+FP}$ (какая часть писем, помеченных как спам, действительно спам)
- **Recall** — $\frac{TP}{TP+FN}$ (какую часть спама мы поймали)
- **F1** — гармоническое среднее precision и recall

In [ ]:
# Предсказания на тесте
test = test.copy()
test['pred'] = test['text'].apply(classify)

# Матрица ошибок (spam — положительный класс)
TP = ((test['label'] == 'spam') & (test['pred'] == 'spam')).sum()
TN = ((test['label'] == 'ham') & (test['pred'] == 'ham')).sum()
FP = ((test['label'] == 'ham') & (test['pred'] == 'spam')).sum()
FN = ((test['label'] == 'spam') & (test['pred'] == 'ham')).sum()

accuracy = (TP + TN) / len(test)
precision = TP / (TP + FP) if (TP + FP) else 0
recall = TP / (TP + FN) if (TP + FN) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print('Метрики качества на тестовой выборке')
print('=' * 60)
print(f'Accuracy:  {accuracy:.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
print(f'F1-score:  {f1:.3f}')

# Визуализация матрицы ошибок
cm = np.array([[TN, FP], [FN, TP]])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'], ax=ax)
ax.set_xlabel('Предсказание')
ax.set_ylabel('Истинный класс')
ax.set_title('Матрица ошибок')
plt.tight_layout()
plt.show()

## 6. Апостериорные вероятности и интерпретация

Посмотрим на распределение апостериорной вероятности $P(\text{spam}\mid\text{текст})$ и на самые «спамные» слова по логарифму отношения вероятностей.

In [ ]:
# Апостериорные вероятности на тесте
test['proba_spam'] = test['text'].apply(lambda t: classify(t, return_proba=True)[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Распределение P(spam) по истинным классам
axes[0].hist(test[test['label'] == 'ham']['proba_spam'], bins=25,
             alpha=0.6, label='истинный ham', color='seagreen')
axes[0].hist(test[test['label'] == 'spam']['proba_spam'], bins=25,
             alpha=0.6, label='истинный spam', color='crimson')
axes[0].axvline(0.5, color='black', linestyle='--', label='порог 0.5')
axes[0].set_xlabel('P(spam | текст)')
axes[0].set_ylabel('Количество писем')
axes[0].set_title('Распределение апостериорной вероятности')
axes[0].legend()

# Самые информативные слова (log-ratio)
sorted_ratio = prob_table.sort_values('log_ratio')
colors = ['crimson' if v > 0 else 'seagreen' for v in sorted_ratio['log_ratio']]
axes[1].barh(sorted_ratio.index, sorted_ratio['log_ratio'], color=colors, alpha=0.7)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('log P(w|spam) / P(w|ham)')
axes[1].set_title('Информативность слов (>0 = спам, <0 = ham)')

plt.tight_layout()
plt.show()

## 7. Выводы

Собираем итоговый отчёт по классификатору.

In [ ]:
print('ИТОГОВЫЙ ОТЧЁТ ПО КЛАССИФИКАТОРУ СПАМА')
print('=' * 60)
print(f'Обучающая выборка: {len(train)} писем')
print(f'Тестовая выборка:  {len(test)} писем')
print(f'Размер словаря:    {V} слов')
print('-' * 60)
print(f'Accuracy:  {accuracy:.1%}')
print(f'Precision: {precision:.1%}')
print(f'Recall:    {recall:.1%}')
print(f'F1-score:  {f1:.3f}')
print('-' * 60)
print('Классификатор построен полностью с нуля на основе теоремы Байеса')
print('и оценки максимального правдоподобия со сглаживанием Лапласа.')

## Упражнения

### Упражнение 1: Влияние сглаживания
1. Как изменится качество при $\alpha = 0.01$ и $\alpha = 10$?
2. Постройте график зависимости F1 от $\alpha$

### Упражнение 2: Порог классификации
1. Постройте ROC-кривую, меняя порог P(spam)
2. Как выбрать порог, если ложная пометка ham как spam особенно нежелательна?

### Упражнение 3: Несбалансированные классы
1. Сгенерируйте данные с соотношением 10% спама и 90% ham
2. Как меняются precision и recall? Почему accuracy обманчива?

### Упражнение 4: Сравнение с библиотекой
1. Обучите `sklearn.naive_bayes.MultinomialNB` на тех же данных
2. Сравните результаты с вашей реализацией

---

**Решения** можно найти в ноутбуке `solutions/17_Solutions.ipynb`